# NC Pi0 Event Displays

Loads the first 100 events from a single NC pi0 ROOT file, runs WC postprocessing
to compute the two-shower intersection vertex, then produces 2D event displays showing:

- Wire-Cell spacepoints (gray dots)
- WC neutrino vertex (blue circle)
- WC primary shower start (cyan triangle)
- WC two-shower intersection vertex (deepskyblue diamond, when available)
- Pandora vertex (red square)
- GLEE vertex (green +)
- Lantern vertex (orange hexagon)
- Truth vertex (black star)
- WC individual shower directions (blue arrows from each PDG=11 particle)

Three projections are shown per event: Z–X (top), Z–Y (side), X–Y (end-on).

In [ ]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../src')))

from src.create_df import _get_file_metadata, _load_chunk
from src.postprocessing import do_wc_postprocessing
from src.file_locations import data_files_location

In [ ]:
filename = "checkout_MCC9.10_Run4c4d5_v10_04_07_13_BNB_NCpi0_overlay_surprise_reco2_hist_4c.root"

meta = _get_file_metadata(filename)
df = _load_chunk(filename, entry_start=0, entry_stop=100, **meta)
print(f"Loaded {len(df)} events")

# Load spacepoint-to-reco-particle mapping (not included in default wc_T_spacepoints_vars)
_f = uproot.open(f"{data_files_location}/{filename}")
_rcids = _f["wcpselection"]["T_spacepoints"].arrays(
    ["Trecchargeblob_spacepoints_real_cluster_id"],
    entry_start=0, entry_stop=100, library="np",
)
df["wc_Trecchargeblob_spacepoints_real_cluster_id"] = list(
    _rcids["Trecchargeblob_spacepoints_real_cluster_id"]
)
del _f, _rcids

In [ ]:
df = do_wc_postprocessing(df)
print("WC postprocessing done")
print("Key added columns: wc_2shw_vtx_x, wc_2shw_vtx_y, wc_2shw_vtx_z, wc_reco_shower_theta, wc_reco_shower_phi")

In [ ]:
# Vertex definitions: label -> (col_x, col_y, col_z, colour, marker, markersize)
VERTEX_STYLES = [
    ("WC vtx",       "wc_reco_nuvtxX",       "wc_reco_nuvtxY",       "wc_reco_nuvtxZ",       "blue",        "o",  80),
    ("WC 2-shw vtx", "wc_2shw_vtx_x",        "wc_2shw_vtx_y",        "wc_2shw_vtx_z",        "deepskyblue", "D",  80),
    ("Pandora vtx",  "pandora_reco_nu_vtx_x", "pandora_reco_nu_vtx_y","pandora_reco_nu_vtx_z","red",         "s",  80),
    ("GLEE vtx",     "glee_reco_vertex_x",    "glee_reco_vertex_y",   "glee_reco_vertex_z",   "green",       "P",  90),
    ("Lantern vtx",  "lantern_vtxX",          "lantern_vtxY",         "lantern_vtxZ",         "orange",      "h",  90),
    ("Truth vtx",    "wc_truth_vtxX",         "wc_truth_vtxY",        "wc_truth_vtxZ",        "black",       "*", 150),
]

SP_COL = {
    "X [cm]": "wc_Trecchargeblob_spacepoints_x",
    "Y [cm]": "wc_Trecchargeblob_spacepoints_y",
    "Z [cm]": "wc_Trecchargeblob_spacepoints_z",
}

# Three projections fill the top-left, top-right, bottom-left cells of a 2×2 grid.
# Bottom-right is reserved for the particle colour legend.
PROJECTIONS = [
    ("Z [cm]", "X [cm]"),  # top view      -> axes[0,0]
    ("Z [cm]", "Y [cm]"),  # side view     -> axes[0,1]
    ("X [cm]", "Y [cm]"),  # end-on view   -> axes[1,0]
]

SHOWER_ENERGY_THRESHOLD_MEV  = 20.0   # min reco shower energy for direction arrows
SP_PARTICLE_KE_THRESH_MEV    = 10.0   # min reco particle KE to get a distinct colour
MAX_DIST_FROM_TRUTH_CM       = 10000.0
CROP_PERCENTILE_LO           = 5.0
CROP_PERCENTILE_HI           = 95.0
CROP_PADDING_FRAC            = 0.05


def _get_wc_showers(row):
    showers = []
    wc_pdg = row["wc_reco_pdg"]
    if isinstance(wc_pdg, float) and np.isnan(wc_pdg):
        return showers
    wc_pdg = np.asarray(wc_pdg)
    wc_startXYZT = row["wc_reco_startXYZT"]
    wc_startMom  = row["wc_reco_startMomentum"]
    for j, pdg in enumerate(wc_pdg):
        if pdg != 11:
            continue
        mom4 = np.asarray(wc_startMom[j], dtype=float)
        if mom4[3] * 1000.0 < SHOWER_ENERGY_THRESHOLD_MEV:
            continue
        start = np.asarray(wc_startXYZT[j][:3], dtype=float)
        mom3  = mom4[:3]
        mag   = np.linalg.norm(mom3)
        if mag > 1e-10:
            showers.append((start, mom3 / mag))
    return showers


def _valid_vertices(row, truth_xyz, has_truth):
    valid = []
    for label, cx, cy, cz, *_ in VERTEX_STYLES:
        vx = row[cx]; vy = row[cy]; vz = row[cz]
        if any(np.isnan(v) for v in (vx, vy, vz)):
            continue
        if abs(vx + 999) < 2 and abs(vy + 999) < 2 and abs(vz + 999) < 2:
            continue
        if has_truth and label != "Truth vtx":
            if np.linalg.norm(np.array([vx, vy, vz]) - truth_xyz) > MAX_DIST_FROM_TRUTH_CM:
                continue
        valid.append((label, vx, vy, vz))
    return valid


def _axis_limits(vals, lo=CROP_PERCENTILE_LO, hi=CROP_PERCENTILE_HI, pad=CROP_PADDING_FRAC):
    if len(vals) < 2:
        return None
    lo_val = np.percentile(vals, lo)
    hi_val = np.percentile(vals, hi)
    margin = (hi_val - lo_val) * pad
    if margin == 0:
        margin = 1.0
    return (lo_val - margin, hi_val + margin)


def _sp_colors_and_legend(row):
    """Return (id_to_color, legend_entries) for spacepoint colouring by reco particle.

    Particles with KE < SP_PARTICLE_KE_THRESH_MEV are mapped to gray and excluded
    from the legend.  Legend is sorted by descending spacepoint count.
    """
    sp_real_ids = np.asarray(row["wc_Trecchargeblob_spacepoints_real_cluster_id"], dtype=float)
    if len(sp_real_ids) == 0:
        return {}, []

    reco_ids  = row["wc_reco_id"]
    reco_pdgs = row["wc_reco_pdg"]
    reco_moms = row["wc_reco_startMomentum"]

    id_to_particle = {}
    if not (isinstance(reco_ids, float) and np.isnan(reco_ids)):
        for j in range(len(reco_ids)):
            rid   = float(reco_ids[j])
            pdg   = int(reco_pdgs[j])
            e_mev = reco_moms[j][3] * 1000.0
            mass  = _MASS_MEV.get(pdg, None)
            ke_mev = (e_mev - mass) if mass is not None else None
            id_to_particle[rid] = (pdg, ke_mev, e_mev)

    # Particles that pass the KE threshold get a distinct tab20 colour
    unique_ids, counts = np.unique(sp_real_ids, return_counts=True)
    order = np.argsort(-counts)
    unique_ids = unique_ids[order]
    counts     = counts[order]

    passing_ids = []
    for uid in unique_ids:
        if uid not in id_to_particle:
            continue
        _, ke_mev, e_mev = id_to_particle[uid]
        effective_ke = ke_mev if ke_mev is not None else e_mev
        if effective_ke >= SP_PARTICLE_KE_THRESH_MEV:
            passing_ids.append(uid)

    cmap = plt.cm.get_cmap("tab20", max(len(passing_ids), 1))
    passing_color = {uid: cmap(i % 20) for i, uid in enumerate(passing_ids)}

    GRAY = (0.5, 0.5, 0.5, 0.4)
    id_to_color = {uid: passing_color.get(uid, GRAY) for uid in unique_ids}

    legend_entries = []
    for uid, cnt in zip(unique_ids, counts):
        if uid not in passing_color:
            continue  # below threshold — omit from legend
        color = passing_color[uid]
        pdg, ke_mev, e_mev = id_to_particle[uid]
        pdg_str = _PDG_LABEL.get(pdg, f"pdg{pdg}")
        energy_str = f"KE={ke_mev:.0f} MeV" if ke_mev is not None else f"E={e_mev:.0f} MeV"
        legend_entries.append((color, f"{pdg_str}  {energy_str}  ({cnt} pts)"))

    return id_to_color, legend_entries


def plot_event_display(row, event_idx):
    run    = row["run"]
    subrun = row["subrun"]
    event  = row["event"]

    sp = {lbl: np.asarray(row[col], dtype=float) for lbl, col in SP_COL.items()}
    has_sp = len(sp["X [cm]"]) > 0

    wc_showers = _get_wc_showers(row)

    truth_xyz = np.array([row["wc_truth_vtxX"], row["wc_truth_vtxY"], row["wc_truth_vtxZ"]], dtype=float)
    has_truth = not np.any(np.isnan(truth_xyz))
    valid_vtx = _valid_vertices(row, truth_xyz, has_truth)

    sp_id_to_color, sp_legend_entries = _sp_colors_and_legend(row)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"Event {event_idx}  |  run {int(run)} subrun {int(subrun)} event {int(event)}\n"
        + get_truth_label(row),
        fontsize=10,
    )

    coord_map = {"X [cm]": 0, "Y [cm]": 1, "Z [cm]": 2}
    ax_positions = [(0, 0), (0, 1), (1, 0)]  # top-left, top-right, bottom-left

    for ax_idx, ((r, c), (xlabel, ylabel)) in enumerate(zip(ax_positions, PROJECTIONS)):
        ax = axes[r, c]

        # ---- spacepoints coloured by reco particle ----
        if has_sp:
            sp_real_ids = np.asarray(row["wc_Trecchargeblob_spacepoints_real_cluster_id"], dtype=float)
            sp_colors = np.array([sp_id_to_color.get(cid, (0.5, 0.5, 0.5, 0.4)) for cid in sp_real_ids])
            ax.scatter(sp[xlabel], sp[ylabel], s=1, c=sp_colors, rasterized=True)

        # ---- vertices ----
        for label, vx, vy, vz in valid_vtx:
            coords = {"X [cm]": vx, "Y [cm]": vy, "Z [cm]": vz}
            _, _, _, _, color, marker, ms = next(s for s in VERTEX_STYLES if s[0] == label)
            ax.scatter(
                [coords[xlabel]], [coords[ylabel]],
                s=ms, c=color, marker=marker, zorder=5,
                label=label if ax_idx == 0 else "",
            )

        # ---- axis limits from 2nd–98th percentile ----
        h_ci = coord_map[xlabel]
        v_ci = coord_map[ylabel]
        h_all = list(sp[xlabel]) if has_sp else []
        v_all = list(sp[ylabel]) if has_sp else []
        for _, vx, vy, vz in valid_vtx:
            h_all.append([vx, vy, vz][h_ci])
            v_all.append([vx, vy, vz][v_ci])

        h_lims = _axis_limits(h_all)
        v_lims = _axis_limits(v_all)
        if h_lims is not None:
            ax.set_xlim(h_lims)
        if v_lims is not None:
            ax.set_ylim(v_lims)

        arrow_length = (h_lims[1] - h_lims[0]) / 10.0 if h_lims is not None else 30.0

        # ---- WC shower direction arrows ----
        for i_shw, (start, direc) in enumerate(wc_showers):
            start_c = {"X [cm]": start[0], "Y [cm]": start[1], "Z [cm]": start[2]}
            direc_c = {"X [cm]": direc[0], "Y [cm]": direc[1], "Z [cm]": direc[2]}
            h0 = start_c[xlabel]; v0 = start_c[ylabel]
            dir2d = np.array([direc_c[xlabel], direc_c[ylabel]])
            mag2d = np.linalg.norm(dir2d)
            if mag2d < 1e-10:
                continue
            dh, dv = dir2d / mag2d * arrow_length
            ax.annotate(
                "",
                xy=(h0 + dh, v0 + dv), xytext=(h0, v0),
                arrowprops=dict(arrowstyle="->", color="blue", lw=1.5),
                zorder=4,
            )
            if ax_idx == 0 and i_shw == 0:
                ax.plot([], [], color="blue", lw=1.5,
                        label=f"WC showers (>{SHOWER_ENERGY_THRESHOLD_MEV:.0f} MeV)")

        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_aspect("equal", adjustable="datalim")
        ax.grid(True, alpha=0.25)

        # Top-left panel: vertex + shower arrow legend
        if ax_idx == 0:
            ax.legend(loc="upper left", fontsize=6.5, markerscale=0.9, framealpha=0.7)

    # ---- Bottom-right: particle colour legend (2 columns) ----
    ax_leg = axes[1, 1]
    ax_leg.axis("off")
    if sp_legend_entries:
        handles = [mpatches.Patch(facecolor=c, label=lbl) for c, lbl in sp_legend_entries]
        ax_leg.legend(
            handles=handles,
            loc="center",
            ncol=2,
            fontsize=7,
            framealpha=0.7,
            title=f"WC reco particles  (KE ≥ {SP_PARTICLE_KE_THRESH_MEV:.0f} MeV)",
            title_fontsize=8,
        )

    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# PDG -> rest mass (MeV/c^2) for common final-state particles
_MASS_MEV = {
    2212:  938.272,   # proton
    2112:  939.565,   # neutron
     211:  139.570,   # pi+
    -211:  139.570,   # pi-
     111:  134.977,   # pi0
      13:  105.658,   # mu-
     -13:  105.658,   # mu+
      11:    0.511,   # e-
     -11:    0.511,   # e+
      22:    0.0,     # photon
     321:  493.677,   # K+
    -321:  493.677,   # K-
     311:  497.648,   # K0
}

# Display symbol for each PDG (matplotlib math strings)
_PDG_LABEL = {
    2212:  r"$p$",
    2112:  r"$n$",
     211:  r"$\pi^+$",
    -211:  r"$\pi^-$",
     111:  r"$\pi^0$",
      13:  r"$\mu^-$",
     -13:  r"$\mu^+$",
      11:  r"$e^-$",
     -11:  r"$e^+$",
      22:  r"$\gamma$",
     321:  r"$K^+$",
    -321:  r"$K^-$",
     311:  r"$K^0$",
}

# Preferred display order for the particle label
_PDG_ORDER = [111, 211, -211, 2212, 2112, 13, -13, 11, -11, 22, 321, -321, 311]

_NEUTRINO_PDGS = {12, -12, 14, -14, 16, -16}

_NU_LABEL = {
    14: r"$\nu_\mu$", -14: r"$\bar{\nu}_\mu$",
    12: r"$\nu_e$",   -12: r"$\bar{\nu}_e$",
    16: r"$\nu_\tau$",-16: r"$\bar{\nu}_\tau$",
}

TRUE_KE_THRESHOLD_MEV = 35.0


def get_truth_label(row, ke_threshold_mev=TRUE_KE_THRESHOLD_MEV):
    """Return a string like 'NC 1π⁰ 2p 1π⁺' describing the true interaction.

    Only primary final-state particles (truth_mother == 0, excluding the
    incoming neutrino) with kinetic energy above *ke_threshold_mev* are counted.
    """
    is_cc  = row["wc_truth_isCC"]
    nu_pdg = row["wc_truth_nuPdg"]

    if isinstance(is_cc, float) and np.isnan(is_cc):
        return "no truth info"

    int_str = "CC" if int(is_cc) else "NC"
    nu_str  = _NU_LABEL.get(int(nu_pdg), f"nu({int(nu_pdg)})")
    in_fv   = bool(row["wc_truth_vtxInside"])

    truth_pdgs = row["wc_truth_pdg"]
    truth_mothers = row["wc_truth_mother"]
    truth_momenta = row["wc_truth_startMomentum"]

    if isinstance(truth_pdgs, float) and np.isnan(truth_pdgs):
        fv_str = " inFV" if in_fv else " outFV"
        return f"{int_str} {nu_str}{fv_str} (no particle list)"

    counts = {}
    other_count = 0
    for j in range(len(truth_pdgs)):
        if truth_mothers[j] != 0:
            continue  # not a primary final-state particle
        pdg = int(truth_pdgs[j])
        if pdg in _NEUTRINO_PDGS:
            continue  # skip the incoming neutrino
        if abs(pdg) >= 1_000_000_000:
            continue  # skip nuclear remnants (e.g. Ar recoil)

        mass_mev = _MASS_MEV.get(pdg, None)
        if mass_mev is None:
            other_count += 1
            continue

        e_mev  = truth_momenta[j][3] * 1000.0
        ke_mev = e_mev - mass_mev
        if ke_mev < ke_threshold_mev:
            continue

        counts[pdg] = counts.get(pdg, 0) + 1

    # Build label in preferred order
    parts = []
    for pdg in _PDG_ORDER:
        if pdg in counts:
            lbl = _PDG_LABEL[pdg]
            parts.append(f"{counts[pdg]}{lbl}")
    # Anything not in the preferred list
    remaining = {k: v for k, v in counts.items() if k not in _PDG_ORDER}
    for pdg, cnt in remaining.items():
        parts.append(f"{cnt}(pdg{pdg})")
    if other_count:
        parts.append(f"{other_count}(other)")

    fv_str = " inFV" if in_fv else " outFV"
    particle_str = " ".join(parts) if parts else "0 particles above threshold"
    return f"{int_str} {nu_str}{fv_str}  {particle_str}  ({ke_threshold_mev:.0f} MeV KE thresh)"

In [ ]:
#for i in range(len(df)):
for i in range(3):
    plot_event_display(df.iloc[i], i)

In [ ]:
def passes_0p_2photon_selection(row, proton_ke_thresh_mev=35.0, photon_e_thresh_mev=100.0):
    """Return True if the event passes generic neutrino selection (wc_kine_reco_Enu > 0),
    has truth vertex in FV, 0 true primary protons (above proton_ke_thresh_mev KE),
    and exactly 2 true photons (PDG 22) with energy above photon_e_thresh_mev."""
    if row["wc_kine_reco_Enu"] <= 0:
        return False

    if not bool(row["wc_truth_vtxInside"]):
        return False

    truth_pdgs    = row["wc_truth_pdg"]
    truth_mothers = row["wc_truth_mother"]
    truth_momenta = row["wc_truth_startMomentum"]

    if isinstance(truth_pdgs, float) and np.isnan(truth_pdgs):
        return False

    n_protons = 0
    n_photons = 0
    for j in range(len(truth_pdgs)):
        pdg = int(truth_pdgs[j])
        e_mev = truth_momenta[j][3] * 1000.0

        # primary protons above KE threshold
        if pdg == 2212 and truth_mothers[j] == 0:
            ke_mev = e_mev - 938.272
            if ke_mev >= proton_ke_thresh_mev:
                n_protons += 1

        # photons above energy threshold (from any parent — captures pi0 daughters)
        if pdg == 22 and e_mev >= photon_e_thresh_mev:
            n_photons += 1

    return n_protons == 0 and n_photons == 2


# Show only generic-selected, inFV events with 0p and exactly 2 true photons > 100 MeV
print("Events passing wc_kine_reco_Enu > 0, inFV, 0p, 2-photon (>100 MeV) selection:")
selected_indices = [i for i in range(len(df)) if passes_0p_2photon_selection(df.iloc[i])]
print(f"  {len(selected_indices)} / {len(df)} events")

for i in selected_indices:
    plot_event_display(df.iloc[i], i)